In [1]:
# ================= CELL 1 =================
import os, cv2, pickle, warnings
import numpy as np
import mediapipe as mp
import tensorflow as tf

from tqdm import tqdm
from collections import deque, Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical

warnings.filterwarnings('ignore')

C:\Users\malai\anaconda3\envs\r-env\Lib\site-packages\h5py\__init__.py:36: UserWarning: h5py is running against HDF5 1.14.3 when it was built against 1.14.2, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "
C:\Users\malai\anaconda3\envs\r-env\Lib\site-packages\tensorflow\python\keras\engine\training_arrays_v1.py:37: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.4.2)
  from scipy.sparse import issparse  # pylint: disable=g-import-not-at-top


In [3]:
# ================= CELL 2 =================
DATASET_PATH = r"C:\Users\malai\Downloads\archive (2)\asl_alphabet_train\asl_alphabet_train"

MODEL_FILE = "sign_model.h5"
ENCODER_FILE = "label_encoder.pkl"

In [5]:
# ================= CELL 3 =================
mp_hands = mp.solutions.hands

hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1
)

X, y = [], []

for label in os.listdir(DATASET_PATH):

    folder = os.path.join(DATASET_PATH, label)

    if not os.path.isdir(folder):
        continue

    for img_name in tqdm(os.listdir(folder), desc=label):

        path = os.path.join(folder, img_name)

        img = cv2.imread(path)

        if img is None:
            continue

        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        result = hands.process(rgb)

        if result.multi_hand_landmarks:

            hand = result.multi_hand_landmarks[0]
            wrist = hand.landmark[0]

            points = []

            for lm in hand.landmark:

                points.extend([
                    lm.x - wrist.x,
                    lm.y - wrist.y,
                    lm.z - wrist.z
                ])

            X.append(points)
            y.append(label)

hands.close()

X = np.array(X)
y = np.array(y)

print("Samples:", len(X))
print("Classes:", np.unique(y))

Z: 100%|██████████| 3000/3000 [02:05<00:00, 23.99it/s]


Samples: 63676
Classes: ['A' 'B' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'J' 'K' 'L' 'M' 'N' 'O' 'P' 'Q' 'R'
 'S' 'T' 'U' 'V' 'W' 'X' 'Y' 'Z' 'del' 'nothing' 'space']


In [7]:
# ================= CELL 4 =================
le = LabelEncoder()

y_encoded = le.fit_transform(y)

y_onehot = to_categorical(y_encoded)

with open(ENCODER_FILE, 'wb') as f:
    pickle.dump(le, f)

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y_onehot,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)

Train: (50940, 63)
Validation: (12736, 63)


In [9]:
# ================= CELL 5 =================
model = Sequential([

    Dense(256, activation='relu', input_shape=(63,)),
    Dropout(0.3),

    Dense(128, activation='relu'),
    Dropout(0.3),

    Dense(len(le.classes_), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │        16,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 29)             │         3,741 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 53,021 (207.11 KB)

 Trainable params: 53,021 (207.11 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# ================= CELL 6 =================
history = model.fit(
    X_train,
    y_train,

    validation_data=(X_val, y_val),

    epochs=20,
    batch_size=64
)

model.save(MODEL_FILE)

print("Model Saved")

Epoch 1/20
796/796 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.5487 - loss: 1.5686 - val_accuracy: 0.9483 - val_loss: 0.2360
Epoch 2/20
796/796 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9273 - loss: 0.2734 - val_accuracy: 0.9644 - val_loss: 0.1375
Epoch 3/20
796/796 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9568 - loss: 0.1725 - val_accuracy: 0.9713 - val_loss: 0.1092
Epoch 4/20
796/796 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9670 - loss: 0.1318 - val_accuracy: 0.9779 - val_loss: 0.0904
Epoch 5/20
796/796 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9704 - loss: 0.1107 - val_accuracy: 0.9818 - val_loss: 0.0793
Epoch 6/20
796/796 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9746 - loss: 0.0969 - val_accuracy: 0.9820 - val_loss: 0.0723
Epoch 7/20
796/796 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9774 - loss: 0.0842 - val_accuracy: 0.9841 - val_loss: 0.0637
Epoch 8/20
796/796 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9800 - loss: 0.0763 - val_accuracy: 0.

Model Saved


In [15]:
# ================= CELL 7 =================
model = load_model(MODEL_FILE)

with open(ENCODER_FILE, 'rb') as f:
    le = pickle.load(f)

cap = cv2.VideoCapture(0)

pred_buffer = deque(maxlen=10)

with mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1
) as hands:

    while cap.isOpened():

        ret, frame = cap.read()

        if not ret:
            break

        frame = cv2.flip(frame, 1)

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        result = hands.process(rgb)

        if result.multi_hand_landmarks:

            hand = result.multi_hand_landmarks[0]

            wrist = hand.landmark[0]

            coords = []

            for lm in hand.landmark:

                coords.extend([
                    lm.x - wrist.x,
                    lm.y - wrist.y,
                    lm.z - wrist.z
                ])

            data = np.array(coords).reshape(1, -1)

            pred = model.predict(data, verbose=0)[0]

            idx = np.argmax(pred)

            pred_buffer.append(idx)

            smooth = Counter(pred_buffer).most_common(1)[0][0]

            label = le.classes_[smooth]

            conf = pred[smooth]

            cv2.putText(
                frame,
                f"{label} ({conf:.2f})",
                (20, 50),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0,255,0),
                2
            )

            mp.solutions.drawing_utils.draw_landmarks(
                frame,
                hand,
                mp_hands.HAND_CONNECTIONS
            )

        cv2.imshow("Sign Language", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()